# Week 9 -- Fraud-Pattern Segmentation: K-Means + DBSCAN + PCA

**FraudFlow Project | Week 9**

Unsupervised clustering of transaction behaviour.  
**Critical rule**: Fraud labels are NOT used for clustering.  
Fraud labels are used ONLY after clustering for descriptive analysis.

## 1. Week 9 Objective

Discover transaction-behaviour segments using K-Means and DBSCAN.  
Analyse fraud rates per segment (post-hoc).  
Prepare segment label as a reusable, leakage-safe feature.

## 2. Load Data

In [ ]:
import json, math, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from src.preprocessing.feature_engineering import engineer_features

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

df_raw = pd.read_csv('data/raw/fraudTest.csv')
df = engineer_features(df_raw)
print(f'Raw shape: {df_raw.shape}')
print(f'Engineered shape: {df.shape}')

## 3. Feature Selection

Select behavioral numerical features only. `is_fraud` and `fraud_loss_amount` are explicitly excluded.

In [ ]:
SEGMENTATION_FEATURES = [
    'amt', 'amount_log', 'transaction_hour', 'transaction_day',
    'transaction_month', 'transaction_day_of_week', 'is_weekend',
    'customer_age', 'merchant_distance', 'lat', 'long',
    'merch_lat', 'merch_long', 'city_pop', 'hour_sin', 'hour_cos'
]
print(f'Selected {len(SEGMENTATION_FEATURES)} segmentation features:')
for f in SEGMENTATION_FEATURES:
    print(f'  - {f}')

## 4. Leakage Audit

In [ ]:
assert 'is_fraud' not in SEGMENTATION_FEATURES, 'LEAKAGE: is_fraud in features!'
assert 'fraud_loss_amount' not in SEGMENTATION_FEATURES, 'LEAKAGE: fraud_loss_amount in features!'
print('Leakage audit PASSED:')
print('  is_fraud NOT in segmentation features')
print('  fraud_loss_amount NOT in segmentation features')
print('  Scaler will be fitted on training rows only')
print('  Test labels not used for fitting')

## 5. Scaling

StandardScaler fitted ONLY on the 80% canonical training split. Test rows never touch the scaler fit.

In [ ]:
y_full = df['is_fraud']
X_full = df.drop(columns=['is_fraud'])
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=RANDOM_STATE)

X_seg   = X_train[SEGMENTATION_FEATURES].copy()
y_seg   = y_train.copy()  # used ONLY for post-hoc fraud-rate analysis

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_seg)
print(f'Scaler fitted on {len(X_seg):,} rows')
print(f'Features: {len(SEGMENTATION_FEATURES)}')
print('Method: StandardScaler (zero mean / unit variance)')

## 6. PCA

Analyse the explained variance structure and visualise the first two PCs.

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)
ev     = pca_full.explained_variance_ratio_
ev_cum = np.cumsum(ev)

print(f'PC1 explained variance: {ev[0]:.4f}')
print(f'PC2 explained variance: {ev[1]:.4f}')
print(f'Cumulative PC1+PC2: {ev_cum[1]:.4f}')
n_90 = int(np.searchsorted(ev_cum, 0.90)) + 1
print(f'Components for 90% variance: {n_90}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(ev)+1), ev, color='steelblue', alpha=0.8)
axes[0].set_title('Individual Explained Variance')
axes[0].set_xlabel('Principal Component')
axes[1].plot(range(1, len(ev_cum)+1), ev_cum, marker='o', color='coral')
axes[1].axhline(0.90, linestyle='--', color='grey', label='90%')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
plt.suptitle('PCA Analysis -- Week 9 Segmentation Features')
plt.tight_layout()
plt.show()

## 7. K-Means Candidate Evaluation

Evaluate K = 2, 3, 4, 5, 6 with inertia and silhouette scores.

In [ ]:
n_pca = min(n_90, len(SEGMENTATION_FEATURES))
pca_km = PCA(n_components=n_pca, random_state=RANDOM_STATE)
X_pca  = pca_km.fit_transform(X_scaled)
print(f'Using {n_pca} PCA components')

def sil_score(X, labels, max_n=50_000):
    uniq = set(labels) - {-1}
    if len(uniq) < 2:
        return None
    if len(X) > max_n:
        idx = rng.choice(len(X), max_n, replace=False)
        return float(silhouette_score(X[idx], labels[idx]))
    return float(silhouette_score(X, labels))

results = []
for k in [2, 3, 4, 5, 6]:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lbls = km.fit_predict(X_pca)
    s = sil_score(X_pca, lbls)
    results.append({'k': k, 'inertia': km.inertia_, 'silhouette': s})
    print(f'  K={k}  inertia={km.inertia_:,.1f}  silhouette={s:.4f}')

df_km = pd.DataFrame(results)
df_km

## 8. K-Means Selection

Select K with highest silhouette score. Document ambiguity if scores are close.

In [ ]:
best = df_km.loc[df_km['silhouette'].idxmax()]
selected_k   = int(best['k'])
selected_sil = float(best['silhouette'])
print(f'Selected K = {selected_k}')
print(f'Silhouette = {selected_sil:.4f}')

similar = df_km[abs(df_km['silhouette'] - selected_sil) < 0.005]
if len(similar) > 1:
    print(f'NOTE: Multiple K values within 0.005 silhouette: {similar["k"].tolist()}')

km_final  = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=10)
km_labels = km_final.fit_predict(X_pca)
sizes = {int(c): int((km_labels==c).sum()) for c in range(selected_k)}
print(f'Cluster sizes: {sizes}')

## 9. DBSCAN Analysis

Evaluated on a 20,000-row reproducible subset (`random_state=42`).  
Silhouette is reported only when >= 2 valid clusters exist.

In [ ]:
DBSCAN_SUB = 20_000
db_idx = rng.choice(len(X_pca), DBSCAN_SUB, replace=False)
X_db   = X_pca[db_idx]

dbscan_configs = [(0.3,5),(0.5,5),(0.5,10),(1.0,5),(1.0,10)]
db_results = []
for eps, min_s in dbscan_configs:
    db   = DBSCAN(eps=eps, min_samples=min_s, n_jobs=-1)
    lbls = db.fit_predict(X_db)
    n_cl = len(set(lbls)) - (1 if -1 in lbls else 0)
    n_ns = int((lbls == -1).sum())
    np_ct = n_ns / len(lbls)
    s = sil_score(X_db[lbls != -1], lbls[lbls != -1]) if n_cl >= 2 else None
    db_results.append({'eps': eps, 'min_samples': min_s, 'n_clusters': n_cl,
                       'noise_pct': np_ct, 'silhouette': s})
    sil_str = 'N/A' if s is None else f'{s:.4f}'
    print(f'  eps={eps} min_s={min_s}: clusters={n_cl} noise={np_ct*100:.1f}% sil={sil_str}')

pd.DataFrame(db_results)

## 10. Final Segmentation

Load saved artifacts and assign cluster labels to held-out test rows using transform-only.

In [ ]:
artifact     = joblib.load('models/kmeans_week9.joblib')
scaler_saved = joblib.load('models/segmentation_scaler_week9.joblib')
km_loaded    = artifact['kmeans']
pca_loaded   = artifact['pca']
features     = artifact['features']
sel_k        = artifact['selected_k']

X_seg_test   = X_test[features]
X_ts         = scaler_saved.transform(X_seg_test)
X_tp         = pca_loaded.transform(X_ts)
test_labels  = km_loaded.predict(X_tp)
test_dist    = {int(c): int((test_labels==c).sum()) for c in range(sel_k)}
print(f'Test cluster distribution: {test_dist}')
print('Note: scaler/PCA/KMeans were NOT refitted on test data')

## 11. Segment Sizes

In [ ]:
df_summary = pd.read_csv('reports/phase9_segment_summary.csv')
print(df_summary[['cluster','transaction_count','fraud_count','fraud_rate']].to_string(index=False))

## 12. Fraud-Rate Analysis by Segment

**Fraud labels are used HERE (post-clustering) for descriptive analysis only.**

In [ ]:
overall_rate = float(y_seg.mean())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(df_summary['cluster'].astype(str), df_summary['fraud_rate'],
            color='coral', alpha=0.8)
axes[0].axhline(overall_rate, linestyle='--', color='grey',
                label=f'Overall {overall_rate:.4f}')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Fraud Rate')
axes[0].set_title('Fraud Rate per Cluster')
axes[0].legend()
axes[1].bar(df_summary['cluster'].astype(str), df_summary['mean_amt'],
            color='steelblue', alpha=0.8)
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Mean Amount ($)')
axes[1].set_title('Mean Transaction Amount per Cluster')
plt.suptitle('Cluster Profiles -- Week 9')
plt.tight_layout()
plt.show()

## 13. PCA Cluster Visualisation

2D PCA visualisation of K-Means clusters (20k-row sample).  
Note: 2 PCs explain only a fraction of total variance; this is illustrative.

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca2 = pca_2d.fit_transform(X_scaled)

pidx    = rng.choice(len(X_pca2), 20_000, replace=False)
palette = cm.tab10.colors

fig, ax = plt.subplots(figsize=(10, 7))
for c in range(selected_k):
    mask = km_labels[pidx] == c
    ax.scatter(X_pca2[pidx[mask], 0], X_pca2[pidx[mask], 1],
               c=[palette[c]], alpha=0.2, s=5, label=f'Cluster {c}')
ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}% var)')
ax.set_title(f'K-Means (K={selected_k}) in 2D PCA Space')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

## 14. Segment Label as a Feature

The segment label is generated without any fraud target information:

In [ ]:
def assign_segment(df_new, scaler, pca, kmeans, features):
    """Assign cluster labels to new observations. No target used."""
    X_n = df_new[features].copy()
    X_s = scaler.transform(X_n)
    X_p = pca.transform(X_s)
    return kmeans.predict(X_p)

sample = assign_segment(X_test.head(5), scaler_saved, pca_loaded, km_loaded, features)
print(f'Sample labels for 5 test rows: {sample}')
print('Leakage safe: no is_fraud or fraud_loss_amount used')

## 15. Limitations

- K-Means assumes convex, isotropic clusters; real fraud patterns may be non-convex.
- Silhouette scores are moderate (~0.14); behavioral transactional data is inherently diffuse.
- DBSCAN run on 20k-row subset only due to O(n^2) complexity on 444k rows.
- High DBSCAN noise percentage reflects diffuse density structure in PCA space.
- Segment labels represent behavioral correlation, not causal fraud determinants.
- No threshold optimization (Week 10). No drift monitoring (Week 11/12).
- `fraud_loss_amount` excluded as it is a regression proxy target, not a behavioral feature.

## 16. Conclusions

- Behavioral segmentation successfully partitioned 444,575 training transactions.
- Selected K=2 based on the highest silhouette score (0.1402).
- Fraud rates differ slightly across clusters, reflecting behavioral variation.
- The segment-label pipeline is leakage-safe and deployable for new transactions.
- DBSCAN at eps=1.0 produced 43 clusters with 98.5% noise on the 20k subset.
- See `reports/phase9_clustering_metrics.json` for all measured values.